# Digital Connectivity & Telehealth Access Equity—Load & Explore

**Agents for Impact 2026 · Digital & Telehealth Equity (`A4I2026-challenge-telehealth`)**

You are building an agent for a **State Broadband & Rural Health Equity Planner**—internal public-health and digital-equity staff deciding **which specific intervention belongs in each underserved neighborhood**:

- **Fixed-broadband last-mile subsidies** where Broadband Serviceable Locations (BSLs) lack `25/3 Mbps` wireline infrastructure,
- **Cellular-enabled tablet lending kits** where cellular coverage exists (`hh_cellular_data_only_count`) but households lack computing devices (`hh_no_computing_device_count`),
- **Community telehealth kiosks** (libraries, FQHCs, county health offices) where clinical shortage coincides with device/affordability gaps, or
- **Mobile physical clinics / non-emergency medical transport** where **both** physical care and connectivity are absent ("double-jeopardy offline deserts") and a video visit is a mirage.

This notebook loads, cleans, and validates your five-table BigQuery foundation. It stops right before creating your **BigQuery ML** (`CREATE MODEL` + `ML.EXPLAIN_PREDICT`) pipeline and your deployed planning agent—that part is yours.

## Plain-words data honesty — what you'll have when this finishes

**CDC PLACES has no "preventable missed care" measure.** Instead, this challenge pairs real empirical CDC PLACES routine checkup prevalence with US Census ACS digital-access counts, organizational CMS NPPES clinic registries, and modeled FCC Broadband Data Collection (BDC) location counts:

| Table | Grain | Source & Provenance | What it tells you |
|---|---|---|---|
| `cdc_tract_health_utilization` | 1 row / census tract | **100% real empirical CDC PLACES (`cwsq-ngmh`)** | Adult routine checkup rate (`routine_checkup_within_1yr_pct`), state baseline (`state_mean_checkup_pct`), within-state shortfall (`checkup_gap_pct_pts`), uninsured, diabetes, hypertension, disability |
| `acs_income_and_device_ownership` | 1 row / census tract | **Real public data** (US Census ACS 5-Year) | Wireline broadband, cellular-data-only, no-internet, no-computing-device, zero-vehicle, poverty counts/rates, `tract_geom`, and post-output `audit_*` shares |
| `fcc_broadband_coverage_by_tract` | 1 row / census tract | **`[Modeled]`** from FCC BDC rural/urban speed-tier benchmarks | Total BSLs, unserved (`<25/3 Mbps`), underserved (`25/3–100/20`), fiber, cable/DSL, fixed wireless, satellite-only counts, ISP count, and `rural_urban_context` |
| `active_health_facilities_registry` | 1 row / Type-2 organization | **Real CMS NPPES (`entity_type_code = 2`) + HRSA HPSA** | Organizational hospitals, FQHCs, RHCs, CAHs (`is_statutory_safety_net_clinic`), HPSA score, and `facility_geom` (**zero** individual clinicians) |
| `historical_facility_closures_10yr` | 1 row / closed facility | **Real UNC Sheps Center & deactivated CMS Type-2 NPIs** | Documented 10-year rural hospital and clinic closures (`2016–2023`) and `closure_geom` |

## The four lanes, and how to spend your 4.5 hours

- **Data lane.** Run this notebook (~2 minutes), inspect the two modeling traps in Section 2, then engineer `v_tract_training_features` and your `CREATE MODEL` + `ML.EXPLAIN_PREDICT` queries from Section 13.
- **Agent lane.** Build a planner agent that turns per-tract `ML.EXPLAIN_PREDICT` feature attributions + spatial clinic/closure distances into concrete, budgeted intervention mixes rather than generic rankings.
- **Front-end lane.** Build an interactive rural-vs-urban planning map/table directly against the five BigQuery tables while the model and agent endpoint come online.
- **Story lane.** Your submission (`GitHub repo`, `Google Skills project`, `pitch deck`, `3-minute video`) is graded asynchronously. Show a tract where naive telehealth deployment fails because households have no internet or computing device—and prove your model used zero `audit_*` inputs.

## Before you start

Two prompts appear the first time you open **Colab Enterprise** in a fresh Google Cloud project, and both are expected:

1. Opening Colab Enterprise prompts you to **enable required APIs**—accept them.
2. The Colab Enterprise header may display a second **"Enable APIs"** banner—enable that too.

Everything this notebook imports (`google-cloud-bigquery`, `pandas`, `pyarrow`, `requests`) is already pre-installed on the standard Colab Enterprise runtime. Do not `pip install` extra packages here.

**How to run:** Pick your `STATE` in the next code cell (default: `"MS"` — Mississippi Delta + Hinds County urban comparator) and select **Runtime → Run all**.

## 1. Setup

Each of the six pre-staged states (`MS`, `AL`, `GA`, `TX`, `CA`, `NY`) pairs **five high-need rural counties** with **one urban comparator county** in the same state so you can contrast rural broadband/closure shocks against urban affordability and uninsured barriers.

In [ ]:
# ---- the one thing you change ------------------------------------------------
STATE    = "MS"               # Choose from STATES below: MS, AL, GA, TX, CA, NY
DATASET  = "a4i_telehealth"   # BigQuery dataset created in your own GCP project
LOCATION = "US"               # Keep in US
# ------------------------------------------------------------------------------

import contextlib
import io
import json
import os
from pathlib import Path
import time
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import requests
from google.cloud import bigquery

STATES = {
    "MS": ("Mississippi", "28"),
    "AL": ("Alabama",     "01"),
    "GA": ("Georgia",     "13"),
    "TX": ("Texas",       "48"),
    "CA": ("California",  "06"),
    "NY": ("New York",    "36"),
}
if STATE not in STATES:
    raise ValueError(f"Unknown STATE={STATE!r}. Choose one of: {list(STATES)}")

STATE_NAME, STATE_FIPS = STATES[STATE]

PROJECT = os.environ.get("GOOGLE_CLOUD_PROJECT")
if not PROJECT:
    PROJECT = bigquery.Client().project
if not PROJECT:
    raise RuntimeError("Could not determine GCP project. Set GOOGLE_CLOUD_PROJECT.")

bq = bigquery.Client(project=PROJECT, location=LOCATION)
ds_ref = bigquery.Dataset(f"{PROJECT}.{DATASET}")
ds_ref.location = LOCATION
bq.create_dataset(ds_ref, exists_ok=True)

print(f"project     : {PROJECT}")
print(f"state       : {STATE_NAME} ({STATE}, state FIPS {STATE_FIPS})")
print(f"dataset     : {PROJECT}.{DATASET} ({LOCATION})")
print(f"pandas      : {pd.__version__}")

### Timing, two-tier validation, and resilient GeoParquet loading

- **`check()` (`FAIL` when false)** — grades **our pipeline**: 11-character `census_tract_geoid` keys, leading-zero FIPS preservation (`AL` = `'01'`, `CA` = `'06'`), 1:1:1 tract alignment across CDC/ACS/FCC, organizational NPI format, and `audit_*` bounds.
- **`note()` (`WARN` up to `limit`, `FAIL` above it)** — flags **dataset characteristics** you need to know before training a BigQuery ML model (such as exact collinearity between two FCC columns and CMS NPPES self-reported hospital taxonomies).

In [ ]:
STEPS, CHECKS = {}, []
NOTEBOOK_START = time.time()

GITHUB_ORG = os.environ.get("A4I_GITHUB_ORG", "Bornism")
REPO_NAME = "A4I2026-challenge-telehealth"
SNAPSHOT_BUCKET = os.environ.get("A4I_SNAPSHOT_BUCKET", "class-demo")
SNAPSHOT_PREFIX = "a4i-2026/challenge-telehealth"
UA = {"User-Agent": "A4I2026-challenge-telehealth-notebook"}


@contextlib.contextmanager
def step(name):
    t0 = time.time()
    print(f"[{name}] ...")
    try:
        yield
    finally:
        dt = time.time() - t0
        STEPS[name] = round(dt, 1)
        print(f"[{name}] {dt:.1f}s")


def _record(name, verdict, detail):
    for c in CHECKS:
        if c["check"] == name:
            c["verdict"], c["detail"] = verdict, detail
            return
    CHECKS.append({"check": name, "verdict": verdict, "detail": detail})


def check(name, ok, detail=""):
    _record(name, "PASS" if ok else "FAIL", detail)
    print(f"  {'PASS' if ok else 'FAIL'}  {name}" + (f": {detail}" if detail else ""))
    return bool(ok)


def note(name, bad, total, detail="", limit=0.01, rows=None):
    frac = (bad / total) if total else 0.0
    verdict = "PASS" if bad == 0 else ("FAIL" if (total and frac > limit) else "WARN")
    msg = detail or (f"{bad:,} of {total:,}" if total else f"{bad:,}")
    _record(name, verdict, msg)
    print(f"  {verdict}  {name}: {msg}")
    if verdict != "PASS" and rows is not None:
        display(rows.head(10))
    return verdict


def safe_str(s, width=None):
    out = s.astype("string")
    if width:
        out = out.str.strip().str.zfill(width)
    return out.where(out.notna(), None)


def spread(series, label=""):
    s = pd.to_numeric(pd.Series(series), errors="coerce").dropna()
    if s.empty:
        return {}
    p10, p50, p90 = s.quantile([0.10, 0.50, 0.90])
    out = {
        "n": int(s.size),
        "min": round(float(s.min()), 3),
        "p10": round(float(p10), 3),
        "median": round(float(p50), 3),
        "p90": round(float(p90), 3),
        "max": round(float(s.max()), 3),
    }
    if label:
        print(f"  {label:<30} " + "  ".join(f"{k}={v}" for k, v in out.items()))
    return out

In [ ]:
def _fetch_parquet_bytes(url, tries=4, timeout=60):
    delay = 1.5
    for attempt in range(tries):
        try:
            resp = requests.get(url, timeout=timeout, headers=UA)
        except requests.RequestException as exc:
            if attempt == tries - 1:
                raise RuntimeError(f"{url} — {type(exc).__name__}: {exc}") from exc
            time.sleep(delay)
            delay *= 2
            continue
        if resp.status_code == 429 or 500 <= resp.status_code < 600:
            if attempt == tries - 1:
                raise RuntimeError(f"{url} — HTTP {resp.status_code} after {tries} tries")
            time.sleep(delay)
            delay *= 2
            continue
        if resp.status_code != 200:
            raise RuntimeError(f"{url} — HTTP {resp.status_code}")
        if resp.content[:4] != b"PAR1":
            raise RuntimeError(f"{url} — response is not a valid Parquet file")
        return resp.content
    raise RuntimeError(f"{url} — exhausted retries")


def _fetch_gcs_parquet_bytes(bucket_name, blob_path):
    from google.cloud import storage as gcs_storage
    client = gcs_storage.Client(project=PROJECT)
    blob = client.bucket(bucket_name).blob(blob_path)
    data = blob.download_as_bytes(timeout=60)
    if data[:4] != b"PAR1":
        raise RuntimeError(f"gs://{bucket_name}/{blob_path} is not a valid Parquet file")
    return data


def read_starter_parquet(filename):
    if not filename or "/" in filename or ".." in filename:
        raise ValueError(f"Invalid starter filename: {filename!r}")
    data_dir = os.environ.get("A4I_DATA_DIR", "").strip()
    local_candidates = (
        ([Path(data_dir) / filename] if data_dir else [])
        + [Path("../data") / filename, Path("data") / filename, Path(filename)]
    )
    for path in local_candidates:
        if path.is_file():
            return pd.read_parquet(path)

    org_candidates = list(dict.fromkeys([GITHUB_ORG, "Bornism", "haggman"]))
    base_override = os.environ.get("A4I_DATA_BASE_URL", "").strip().rstrip("/")
    urls = ([f"{base_override}/{filename}"] if base_override else []) + [
        f"https://raw.githubusercontent.com/{org}/{REPO_NAME}/main/data/{filename}"
        for org in org_candidates
    ]
    errors = []
    for url in urls:
        try:
            return pd.read_parquet(io.BytesIO(_fetch_parquet_bytes(url)))
        except RuntimeError as exc:
            errors.append(str(exc))

    if SNAPSHOT_BUCKET:
        blob_path = f"{SNAPSHOT_PREFIX}/starter/{filename}"
        try:
            return pd.read_parquet(io.BytesIO(_fetch_gcs_parquet_bytes(SNAPSHOT_BUCKET, blob_path)))
        except Exception as exc:  # noqa: BLE001
            errors.append(f"gs://{SNAPSHOT_BUCKET}/{blob_path} ({type(exc).__name__}: {exc})")

    raise RuntimeError(f"Could not locate {filename} locally or remotely: {'; '.join(errors)}")

def _to_geoparquet_buffer(df, geom_col=None):
    table = pa.Table.from_pandas(df, preserve_index=False)
    if geom_col and geom_col in df.columns:
        geo_meta = {
            "version": "1.0.0",
            "primary_column": geom_col,
            "columns": {
                geom_col: {
                    "encoding": "WKB",
                    "geometry_types": [],
                    "crs": None,
                    "orientation": "counterclockwise",
                    "edges": "spherical",
                }
            },
        }
        meta = dict(table.schema.metadata or {})
        meta[b"geo"] = json.dumps(geo_meta).encode("utf-8")
        table = table.replace_schema_metadata(meta)
    buf = io.BytesIO()
    pq.write_table(table, buf)
    buf.seek(0)
    return buf


def load_table(df, table_name, geom_col=None, description=""):
    if df is None or not table_name:
        raise ValueError("df and table_name are required")
    ref = f"{PROJECT}.{DATASET}.{table_name}"
    buf = _to_geoparquet_buffer(df, geom_col=geom_col)
    cfg = bigquery.LoadJobConfig(
        source_format=bigquery.SourceFormat.PARQUET,
        write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
    )
    bq.load_table_from_file(buf, ref, job_config=cfg).result()
    tbl = bq.get_table(ref)
    if description:
        tbl.description = description
        bq.update_table(tbl, ["description"])
    print(f"  loaded {table_name:<38} {tbl.num_rows:>6,} rows")
    return tbl

## 2. Start with the wrong answer

Teams rushing into **BigQuery ML** on this dataset make two mistakes inside the first twenty minutes—and both look great on a slide until a judge asks what the model actually learned:

1. **Pooling raw `routine_checkup_within_1yr_pct` across states without centering:** CDC PLACES derives checkup prevalence from state BRFSS telephone surveys, whose state baselines differ by nearly **14 percentage points** (`Mississippi` mean `83.6%` vs. `California` mean `70.0%`). Pool raw checkup rates across states and `no_internet_share` correlates **positively** (`r = +0.32`) with getting a checkup—implying that disconnecting people's internet improves preventive care! That is why `checkup_gap_pct_pts = (state_mean_checkup_pct - routine_checkup_within_1yr_pct) * 100` centers every tract against its own state baseline (`> 0` means the tract falls *below* its state average).
2. **Target leakage + exact collinearity from `SELECT *`:** Leave `routine_checkup_within_1yr_pct` and `state_mean_checkup_pct` in your training query when predicting `checkup_gap_pct_pts`, and `LINEAR_REG` achieves **`R^2 = 1.000`** because it just rediscovered subtraction—assigning **zero** weight to broadband, devices, or clinic distance! Worse, in `fcc_broadband_coverage_by_tract`, `unserved_bsl_under_25_3_count` and `satellite_only_bsl_count` are **100% identical** on every row, splitting feature attributions arbitrarily if both stay in the feature matrix.

In [ ]:
with step("section 2 baseline-confound & leakage demonstration"):
    _all_cdc = read_starter_parquet("cdc_tract_health_utilization.parquet")
    _all_acs = read_starter_parquet("acs_income_and_device_ownership.parquet")
    _all_fcc = read_starter_parquet("fcc_broadband_coverage_by_tract.parquet")
    _pooled = _all_cdc.merge(_all_acs, on=["census_tract_geoid", "state_code"]).merge(
        _all_fcc, on=["census_tract_geoid", "state_code"]
    )
    _pooled["no_internet_share"] = (
        _pooled["hh_no_internet_access_count"] / _pooled["total_households"].clip(lower=1)
    )

    state_baselines = (
        _pooled.groupby("state_code")
        .agg(
            tracts=("census_tract_geoid", "count"),
            mean_raw_checkup_pct=("routine_checkup_within_1yr_pct", lambda s: round(s.mean() * 100, 1)),
            mean_no_internet_pct=("no_internet_share", lambda s: round(s.mean() * 100, 1)),
        )
        .reset_index()
    )
    print("Cross-state BRFSS baseline spread (why raw checkup % confounds across states):")
    display(state_baselines)

    r_raw = _pooled["routine_checkup_within_1yr_pct"].corr(_pooled["no_internet_share"])
    exact_dup = int(
        (_pooled["unserved_bsl_under_25_3_count"] == _pooled["satellite_only_bsl_count"]).sum()
    )
    formula_max_err = float(
        (
            (_pooled["state_mean_checkup_pct"] - _pooled["routine_checkup_within_1yr_pct"]) * 100.0
            - _pooled["checkup_gap_pct_pts"]
        )
        .abs()
        .max()
    )
    print(f"Pooled correlation(raw checkup %, no_internet_share) : {r_raw:+.3f}  <-- Simpson's paradox!")
    print(f"Max |state_mean - raw_checkup - checkup_gap|         : {formula_max_err:.4f} pct pts  <-- 100% target leakage if left in SELECT *!")
    print(f"Rows where unserved_bsl == satellite_only_bsl        : {exact_dup}/{len(_pooled)}  <-- exact collinearity!")

## 3. Choosing a state

Every state below includes **five rural high-need counties** plus **one urban comparator county** (`18` urban comparator tracts per state). Watch two facts when choosing:

- **Leading-zero FIPS (`AL` = `'01'`, `CA` = `'06'`):** Keep `census_tract_geoid` (11 chars) and `county_fips` (5 chars) as strings everywhere.
- **Closure density (`MS` vs. `GA` / `AL` / `NY`):** Mississippi's Delta catchment carries **18** documented 10-year facility closures (the richest closure-shock signal in the pack), whereas Georgia has `2` and Alabama/New York each have `3`.

| State | `FIPS` | Rural counties + Urban comparator | Tracts (`Rural` + `Urban`) | Unserved BSLs (`<25/3`) | No home internet | Active facilities (Safety-net) | 10-yr closures |
|---|---|---|---:|---:|---:|---:|---:|
| **`MS`** *(default)* | `28` | Leflore, Coahoma, Bolivar, Washington, Sunflower + *Hinds* | 65 (`47` + `18`) | 21.3% | 14.7% | 90 (`54`) | 18 |
| `AL` | `01` | Dallas, Wilcox, Perry, Lowndes, Sumter + *Jefferson* | 43 (`25` + `18`) | 15.6% | 10.4% | 89 (`53`) | 3 |
| `GA` | `13` | Dougherty, Terrell, Crisp, Ware, Emanuel + *Fulton* | 57 (`39` + `18`) | 20.6% | 12.9% | 90 (`54`) | 2 |
| `TX` | `48` | Hidalgo, Starr, Zavala, Cherokee, Nacogdoches + *Dallas* | 72 (`54` + `18`) | 19.0% | 11.0% | 90 (`54`) | 6 |
| `CA` | `06` | Madera, Merced, Fresno, Imperial, Shasta + *Santa Clara* | 101 (`83` + `18`) | 19.0% | 6.4% | 90 (`54`) | 6 |
| `NY` | `36` | Niagara, Chautauqua, St. Lawrence, Franklin, Delaware + *Bronx* | 90 (`72` + `18`) | 19.7% | 9.2% | 90 (`54`) | 3 |

## 4. Empirical health utilization (`cdc_tract_health_utilization`)

All columns here come from **CDC PLACES (`cwsq-ngmh`)** tract surveillance (`measurement_year = 2023`):

- **`checkup_gap_pct_pts` (target variable):** `(state_mean_checkup_pct - routine_checkup_within_1yr_pct) * 100`. **Positive values mean a tract has a *lower* checkup rate than its state average** (larger preventive-care gap).
- **Important empirical nuance:** Notice in the summary below that older, high-chronic-burden rural tracts (`diabetes_pct`, `hypertension_pct`) often report slightly *higher* annual checkup rates than younger uninsured urban tracts, even when rural residents travel 20+ km after a hospital closure! That means preventive checkup gap and chronic-care access burden are **distinct dimensions**—and your planner agent gets extra credit for noticing rather than flattening them.

In [ ]:
with step("load cdc_tract_health_utilization"):
    raw_cdc = read_starter_parquet("cdc_tract_health_utilization.parquet")
    cdc = raw_cdc[raw_cdc["state_code"] == STATE].copy()
    cdc["census_tract_geoid"] = safe_str(cdc["census_tract_geoid"], width=11)

    spread(cdc["routine_checkup_within_1yr_pct"], "routine_checkup_within_1yr_pct")
    spread(cdc["checkup_gap_pct_pts"], "checkup_gap_pct_pts (+ = worse)")
    spread(cdc["uninsured_adults_pct"], "uninsured_adults_pct")

    load_table(
        cdc,
        "cdc_tract_health_utilization",
        description=(
            f"Real CDC PLACES (cwsq-ngmh) adult routine checkup rate, within-state checkup "
            f"gap (pct pts), uninsured share, and chronic disease prevalence for {STATE_NAME}."
        ),
    )

## 5. Household internet, devices, vehicles & income (`acs_income_and_device_ownership`)

From the **US Census Bureau ACS 5-Year** estimates, this table separates **three very different household barriers** that call for completely different interventions:

1. **`hh_no_internet_access_count`** — no home internet subscription of any kind.
2. **`hh_cellular_data_only_count`** — smartphone/cellular plan only, no home wireline fiber/cable.
3. **`hh_no_computing_device_count`** — no desktop, laptop, tablet, or smartphone capable of a telehealth visit.

Convert raw household counts into shares (`SAFE_DIVIDE(count, total_households)`) before feeding them to `CREATE MODEL` so large-population tracts do not dominate solely by size. Keep `audit_*` columns strictly out of training features (Section 11).

In [ ]:
with step("load acs_income_and_device_ownership"):
    raw_acs = read_starter_parquet("acs_income_and_device_ownership.parquet")
    acs = raw_acs[raw_acs["state_code"] == STATE].copy()
    acs["census_tract_geoid"] = safe_str(acs["census_tract_geoid"], width=11)
    acs["county_fips"] = safe_str(acs["county_fips"], width=5)

    spread(acs["hh_no_internet_access_count"] / acs["total_households"].clip(lower=1), "no_internet_share")
    spread(acs["hh_cellular_data_only_count"] / acs["total_households"].clip(lower=1), "cellular_only_share")
    spread(acs["hh_no_computing_device_count"] / acs["total_households"].clip(lower=1), "no_device_share")
    spread(acs["no_vehicle_rate"], "no_vehicle_rate")

    load_table(
        acs,
        "acs_income_and_device_ownership",
        geom_col="tract_geom",
        description=(
            f"US Census ACS 5-Year household internet, cellular-only, no-device, no-vehicle, "
            f"poverty, tract_geom (GEOGRAPHY), and audit_* shares for {STATE_NAME}."
        ),
    )

## 6. Modeled broadband serviceable locations (`fcc_broadband_coverage_by_tract`)

Labeled **`[Modeled]`** in `data_provenance`: tract-level Broadband Serviceable Location (BSL) counts calibrated to **Federal Communications Commission (FCC) Broadband Data Collection (BDC)** rural and urban speed-tier benchmarks:

- **Unserved (`<25/3 Mbps`):** `unserved_bsl_under_25_3_count` — cannot sustain reliable two-way HD video telehealth.
- **Underserved (`25/3` to `<100/20 Mbps`):** `underserved_bsl_25_3_to_100_20_count` — struggles when multiple household members share bandwidth.
- **Collinearity reminder:** As shown in Section 2, `satellite_only_bsl_count` equals `unserved_bsl_under_25_3_count` across this modeled table—use **one** (`unserved_bsl_share`), never both, in `LINEAR_REG`.

In [ ]:
with step("load fcc_broadband_coverage_by_tract"):
    raw_fcc = read_starter_parquet("fcc_broadband_coverage_by_tract.parquet")
    fcc = raw_fcc[raw_fcc["state_code"] == STATE].copy()
    fcc["census_tract_geoid"] = safe_str(fcc["census_tract_geoid"], width=11)
    fcc["county_fips"] = safe_str(fcc["county_fips"], width=5)

    ctx_summary = (
        fcc.assign(unserved_share=fcc["unserved_bsl_under_25_3_count"] / fcc["total_bsl_locations"].clip(lower=1))
        .groupby("rural_urban_context")
        .agg(
            tracts=("census_tract_geoid", "count"),
            mean_unserved_pct=("unserved_share", lambda s: round(s.mean() * 100, 1)),
            median_isps=("isp_provider_count", "median"),
        )
        .reset_index()
    )
    display(ctx_summary)

    load_table(
        fcc,
        "fcc_broadband_coverage_by_tract",
        description=(
            f"[Modeled] Tract Broadband Serviceable Location (BSL) speed-tier and technology "
            f"counts calibrated to FCC BDC benchmarks for {STATE_NAME}."
        ),
    )

## 7. Active organizational health facilities (`active_health_facilities_registry`)

Built from **CMS National Plan and Provider Enumeration System (NPPES) Type-2 organizational NPIs** (`entity_type_code = 2`; **zero** individual practitioners) enriched with **HRSA Health Professional Shortage Area (HPSA)** primary care scores:

- **`is_statutory_safety_net_clinic = TRUE`** marks **Federally Qualified Health Centers (FQHCs)**, **Rural Health Clinics (RHCs)**, and **Critical Access Hospitals (CAHs)**.
- **Real federal data quirk worth knowing:** CMS NPPES lets organizations self-report their taxonomy code (`282NC0060X` = Critical Access Hospital). In several states (notably New York), major urban academic hospitals self-classified under `282NC0060X` in NPPES! Section 12 surfaces those via `note()` so your team isn't baffled seeing an urban medical center labeled `'Critical Access Hospital (CAH)'`.

In [ ]:
with step("load active_health_facilities_registry"):
    raw_fac = read_starter_parquet("active_health_facilities_registry.parquet")
    facilities = raw_fac[raw_fac["state_code"] == STATE].copy()
    facilities["facility_npi"] = safe_str(facilities["facility_npi"], width=10)
    facilities["zip_code"] = safe_str(facilities["zip_code"], width=5)

    display(
        facilities.groupby(["facility_type", "is_statutory_safety_net_clinic"])
        .agg(facilities=("facility_npi", "count"), mean_hpsa=("hpsa_primary_care_score", "mean"))
        .reset_index()
    )

    load_table(
        facilities,
        "active_health_facilities_registry",
        geom_col="facility_geom",
        description=(
            f"CMS NPPES active Type-2 organizational facilities + HRSA HPSA scores and "
            f"facility_geom (GEOGRAPHY) for {STATE_NAME}. Zero individual clinicians."
        ),
    )

## 8. Ten-year facility closures (`historical_facility_closures_10yr`)

Combines publicly documented **UNC Cecil G. Sheps Center** rural hospital closures with deactivated **CMS NPPES Type-2 organizational facility records** (`2016–2023`). When a rural hospital or FQHC closes, surrounding tracts lose in-person primary and urgent care—making telehealth tempting on paper right where broadband and device ownership are weakest.

In [ ]:
with step("load historical_facility_closures_10yr"):
    raw_closures = read_starter_parquet("historical_facility_closures_10yr.parquet")
    closures = raw_closures[raw_closures["state_code"] == STATE].copy()
    closures["npi"] = safe_str(closures["npi"], width=10)

    display(
        closures[
            ["closure_event_id", "county_name", "city", "facility_name", "prior_facility_type", "closure_year"]
        ].head(10)
    )

    load_table(
        closures,
        "historical_facility_closures_10yr",
        geom_col="closure_geom",
        description=(
            f"Historical 10-year organizational hospital and clinic closures (UNC Sheps Center "
            f"& deactivated CMS Type-2 NPIs) with closure_geom (GEOGRAPHY) for {STATE_NAME}."
        ),
    )

## 9. Inventory what landed in BigQuery

In [ ]:
with step("inventory loaded BigQuery tables"):
    inv = bq.query(f"""
        SELECT
            table_id,
            row_count,
            ROUND(size_bytes / 1024.0, 1) AS size_kb
        FROM `{PROJECT}.{DATASET}.__TABLES__`
        ORDER BY table_id
    """).to_dataframe()
    display(inv)

## 10. Double-jeopardy offline health deserts — spatial + digital payoff query

Using native BigQuery `GEOGRAPHY` columns (`ST_DISTANCE(ST_CENTROID(a.tract_geom), f.facility_geom)`), let's measure for every tract in `{STATE}`:

1. Distance in km to the nearest **active clinic** and nearest **statutory safety-net clinic**,
2. Distance in km to the nearest **10-year closed hospital/clinic**, and
3. How physical distance collides with **unserved BSL share (`<25/3 Mbps`)**, **no home internet**, and **no computing device** across `Rural` vs. `Urban Comparator` tracts.

In [ ]:
with step("spatial + digital double-jeopardy payoff query"):
    access_summary = bq.query(f"""
        WITH clinic_dist AS (
            SELECT
                a.census_tract_geoid,
                ROUND(MIN(ST_DISTANCE(ST_CENTROID(a.tract_geom), f.facility_geom)) / 1000.0, 2)
                    AS nearest_active_clinic_km,
                ROUND(MIN(IF(f.is_statutory_safety_net_clinic,
                             ST_DISTANCE(ST_CENTROID(a.tract_geom), f.facility_geom), NULL)) / 1000.0, 2)
                    AS nearest_safety_net_km
            FROM `{PROJECT}.{DATASET}.acs_income_and_device_ownership` AS a
            INNER JOIN `{PROJECT}.{DATASET}.active_health_facilities_registry` AS f
                ON a.state_code = f.state_code
            GROUP BY 1
        ),
        closure_dist AS (
            SELECT
                a.census_tract_geoid,
                ROUND(MIN(ST_DISTANCE(ST_CENTROID(a.tract_geom), c.closure_geom)) / 1000.0, 2)
                    AS nearest_closure_km
            FROM `{PROJECT}.{DATASET}.acs_income_and_device_ownership` AS a
            LEFT JOIN `{PROJECT}.{DATASET}.historical_facility_closures_10yr` AS c
                ON a.state_code = c.state_code
            GROUP BY 1
        )
        SELECT
            b.rural_urban_context,
            COUNT(*) AS tracts,
            ROUND(AVG(d.nearest_active_clinic_km), 1) AS avg_clinic_km,
            ROUND(AVG(d.nearest_safety_net_km), 1) AS avg_safety_net_km,
            ROUND(AVG(cl.nearest_closure_km), 1) AS avg_closure_km,
            ROUND(AVG(SAFE_DIVIDE(b.unserved_bsl_under_25_3_count, b.total_bsl_locations)) * 100, 1)
                AS avg_unserved_bsl_pct,
            ROUND(AVG(SAFE_DIVIDE(a.hh_no_internet_access_count, a.total_households)) * 100, 1)
                AS avg_no_internet_pct,
            ROUND(AVG(SAFE_DIVIDE(a.hh_no_computing_device_count, a.total_households)) * 100, 1)
                AS avg_no_device_pct,
            ROUND(AVG(u.checkup_gap_pct_pts), 2) AS avg_checkup_gap_pts
        FROM `{PROJECT}.{DATASET}.cdc_tract_health_utilization` AS u
        INNER JOIN `{PROJECT}.{DATASET}.acs_income_and_device_ownership` AS a USING (census_tract_geoid)
        INNER JOIN `{PROJECT}.{DATASET}.fcc_broadband_coverage_by_tract` AS b USING (census_tract_geoid)
        INNER JOIN clinic_dist AS d USING (census_tract_geoid)
        LEFT JOIN closure_dist AS cl USING (census_tract_geoid)
        GROUP BY 1
        ORDER BY 1
    """).to_dataframe()
    display(access_summary)

## 11. Deliberate exclusions & the Post-Output Equity Audit rule

**Never include `audit_black_share`, `audit_hispanic_share`, `audit_asian_share`, `audit_indigenous_share`, or `audit_white_share` in `v_tract_training_features` or `CREATE MODEL`.**

Why: if demographic race/ethnicity shares enter your `CREATE MODEL` feature list, `ML.EXPLAIN_PREDICT` attributes preventive-care gaps to a neighborhood's racial composition rather than to **actionable infrastructure barriers** (missing fiber, missing tablets, uninsured rates, clinic distance)—and risks baking historical underinvestment into allocation thresholds.

Instead:
1. Train and explain strictly on **connectivity, device, affordability, insurance, and clinic-distance features**.
2. Join `audit_*` and `no_vehicle_rate` **after** generating your intervention plan to verify whether high-need Black, Hispanic, Indigenous, and zero-vehicle communities receive equitable physical vs. digital investment.

In [ ]:
with step("post-output equity audit baseline"):
    equity_view = bq.query(f"""
        SELECT
            b.rural_urban_context,
            COUNT(*) AS tracts,
            ROUND(AVG(a.poverty_rate) * 100, 1) AS avg_poverty_pct,
            ROUND(AVG(a.no_vehicle_rate) * 100, 1) AS avg_no_vehicle_pct,
            ROUND(AVG(a.audit_black_share) * 100, 1) AS audit_black_pct,
            ROUND(AVG(a.audit_hispanic_share) * 100, 1) AS audit_hispanic_pct,
            ROUND(AVG(a.audit_indigenous_share) * 100, 1) AS audit_indigenous_pct
        FROM `{PROJECT}.{DATASET}.acs_income_and_device_ownership` AS a
        INNER JOIN `{PROJECT}.{DATASET}.fcc_broadband_coverage_by_tract` AS b USING (census_tract_geoid)
        GROUP BY 1
        ORDER BY 1
    """).to_dataframe()
    display(equity_view)

## 12. Validation — graded against the loaded BigQuery tables

In [ ]:
with step("validate loaded BigQuery tables"):
    S = f"{PROJECT}.{DATASET}"

    u_stats = list(bq.query(f"""
        SELECT
            COUNT(*) AS n_cdc,
            COUNT(DISTINCT census_tract_geoid) AS n_distinct,
            COUNTIF(LENGTH(census_tract_geoid) != 11) AS bad_geoid_len,
            COUNTIF(SUBSTR(census_tract_geoid, 1, 2) != '{STATE_FIPS}') AS bad_fips_prefix,
            COUNTIF(state_code != '{STATE}') AS wrong_state,
            COUNTIF(routine_checkup_within_1yr_pct <= 0 OR routine_checkup_within_1yr_pct >= 1) AS bad_chk_range,
            MAX(ABS((state_mean_checkup_pct - routine_checkup_within_1yr_pct) * 100.0 - checkup_gap_pct_pts)) AS max_gap_diff
        FROM `{S}.cdc_tract_health_utilization`
    """).result())[0]

    check("cdc: >= 40 tracts loaded", u_stats.n_cdc >= 40, f"{u_stats.n_cdc} tracts")
    check("cdc: unique census_tract_geoid", u_stats.n_cdc == u_stats.n_distinct, f"{u_stats.n_distinct} distinct")
    check("cdc: all tract IDs are 11 chars (leading zero intact)", u_stats.bad_geoid_len == 0, f"{u_stats.bad_geoid_len} bad")
    check("cdc: all tracts match STATE_FIPS", u_stats.bad_fips_prefix == 0, f"expected {STATE_FIPS}")
    check("cdc: all rows match STATE", u_stats.wrong_state == 0, f"{u_stats.wrong_state} wrong")
    check("cdc: routine_checkup_within_1yr_pct in (0, 1)", u_stats.bad_chk_range == 0, f"{u_stats.bad_chk_range} bad")
    check("cdc: checkup_gap_pct_pts matches (state_mean - tract)*100", (u_stats.max_gap_diff or 0.0) < 0.05, f"max diff={u_stats.max_gap_diff:.4f}")

    af_stats = list(bq.query(f"""
        SELECT
            (SELECT COUNT(*) FROM `{S}.acs_income_and_device_ownership`) AS n_acs,
            (SELECT COUNT(*) FROM `{S}.fcc_broadband_coverage_by_tract`) AS n_fcc,
            (SELECT COUNT(*)
               FROM `{S}.cdc_tract_health_utilization` u
               INNER JOIN `{S}.acs_income_and_device_ownership` a USING (census_tract_geoid)
               INNER JOIN `{S}.fcc_broadband_coverage_by_tract` b USING (census_tract_geoid)) AS n_joined,
            (SELECT COUNTIF(total_households <= 0 OR hh_no_internet_access_count > total_households)
               FROM `{S}.acs_income_and_device_ownership`) AS bad_hh,
            (SELECT COUNTIF(
                audit_black_share < 0 OR audit_black_share > 1
                OR audit_hispanic_share < 0 OR audit_hispanic_share > 1
                OR audit_asian_share < 0 OR audit_asian_share > 1
                OR audit_indigenous_share < 0 OR audit_indigenous_share > 1
                OR audit_white_share < 0 OR audit_white_share > 1
            ) FROM `{S}.acs_income_and_device_ownership`) AS bad_audit,
            (SELECT COUNTIF(rural_urban_context = 'Rural') FROM `{S}.fcc_broadband_coverage_by_tract`) AS n_rural,
            (SELECT COUNTIF(rural_urban_context = 'Urban Comparator') FROM `{S}.fcc_broadband_coverage_by_tract`) AS n_urban,
            (SELECT COUNTIF(unserved_bsl_under_25_3_count = satellite_only_bsl_count)
               FROM `{S}.fcc_broadband_coverage_by_tract`) AS n_collinear
    """).result())[0]

    check("acs & fcc: 1:1:1 tract alignment with cdc", af_stats.n_acs == u_stats.n_cdc and af_stats.n_fcc == u_stats.n_cdc and af_stats.n_joined == u_stats.n_cdc, f"cdc={u_stats.n_cdc}, acs={af_stats.n_acs}, fcc={af_stats.n_fcc}, joined={af_stats.n_joined}")
    check("acs: valid household & internet counts", af_stats.bad_hh == 0, f"{af_stats.bad_hh} invalid")
    check("acs: audit_* shares within [0, 1]", af_stats.bad_audit == 0, f"{af_stats.bad_audit} out of bounds")
    check("fcc: both Rural and Urban Comparator tracts present", af_stats.n_rural >= 20 and af_stats.n_urban >= 15, f"rural={af_stats.n_rural}, urban={af_stats.n_urban}")

    note(
        "fcc modeled collinearity: unserved_bsl_under_25_3_count == satellite_only_bsl_count (drop one in BQML)",
        af_stats.n_collinear,
        af_stats.n_fcc,
        detail=f"{af_stats.n_collinear}/{af_stats.n_fcc} identical — include only unserved_bsl_share in CREATE MODEL",
        limit=1.01,
    )

    fac_stats = list(bq.query(rf"""
        SELECT
            COUNT(*) AS n_fac,
            COUNT(DISTINCT facility_npi) AS n_distinct,
            COUNTIF(NOT REGEXP_CONTAINS(facility_npi, r'^\d{{10}}$')) AS bad_npi,
            COUNTIF(state_code != '{STATE}') AS wrong_state,
            COUNTIF(is_statutory_safety_net_clinic) AS n_snet,
            COUNTIF(hpsa_primary_care_score < 0 OR hpsa_primary_care_score > 26) AS bad_hpsa,
            (SELECT COUNT(*) FROM `{S}.historical_facility_closures_10yr`) AS n_closures,
            (SELECT COUNTIF(state_code != '{STATE}') FROM `{S}.historical_facility_closures_10yr`) AS bad_closure_state,
            (SELECT COUNTIF(closure_year < 2014 OR closure_year > 2026) FROM `{S}.historical_facility_closures_10yr`) AS bad_closure_yr
        FROM `{S}.active_health_facilities_registry`
    """).result())[0]

    check("facilities: >= 80 active organizational facilities", fac_stats.n_fac >= 80, f"{fac_stats.n_fac} facilities")
    check("facilities: unique 10-digit organizational NPIs", fac_stats.n_fac == fac_stats.n_distinct and fac_stats.bad_npi == 0, f"{fac_stats.n_distinct} distinct, {fac_stats.bad_npi} malformed")
    check("facilities: all rows match STATE", fac_stats.wrong_state == 0, f"{fac_stats.wrong_state} wrong")
    check("facilities: >= 30 statutory safety-net clinics", fac_stats.n_snet >= 30, f"{fac_stats.n_snet} safety-net clinics")
    check("facilities: HPSA primary care scores in [0, 26]", fac_stats.bad_hpsa == 0, f"{fac_stats.bad_hpsa} out of range")
    check("closures: >= 2 historical closures in STATE", fac_stats.n_closures >= 2, f"{fac_stats.n_closures} closures")
    check("closures: all closure rows match STATE and valid year", fac_stats.bad_closure_state == 0 and fac_stats.bad_closure_yr == 0, f"wrong_state={fac_stats.bad_closure_state}, bad_yr={fac_stats.bad_closure_yr}")

    urban_cah = bq.query(f"""
        SELECT facility_name, county_name, city, taxonomy_code
        FROM `{S}.active_health_facilities_registry`
        WHERE taxonomy_code = '282NC0060X'
          AND county_name IN (
              SELECT DISTINCT county_name
              FROM `{S}.fcc_broadband_coverage_by_tract`
              WHERE rural_urban_context = 'Urban Comparator'
          )
    """).to_dataframe()
    if len(urban_cah):
        note(
            "CMS NPPES self-reported taxonomy: urban comparator hospitals self-classified as 282NC0060X (CAH)",
            len(urban_cah),
            fac_stats.n_fac,
            detail=f"{len(urban_cah)} of {fac_stats.n_fac} facilities in urban comparator county carry CAH taxonomy",
            limit=0.30,
            rows=urban_cah,
        )

    geo_cols = bq.query(f"""
        SELECT table_name, column_name, data_type
        FROM `{S}.INFORMATION_SCHEMA.COLUMNS`
        WHERE (table_name = 'acs_income_and_device_ownership' AND column_name = 'tract_geom')
           OR (table_name = 'active_health_facilities_registry' AND column_name = 'facility_geom')
           OR (table_name = 'historical_facility_closures_10yr' AND column_name = 'closure_geom')
    """).to_dataframe()
    n_geog = int((geo_cols["data_type"] == "GEOGRAPHY").sum()) if len(geo_cols) else 0
    check("spatial: tract_geom, facility_geom, and closure_geom landed as native GEOGRAPHY", n_geog == 3, f"{n_geog}/3 GEOGRAPHY columns")

    failed = [c for c in CHECKS if c["verdict"] == "FAIL"]
    warned = [c for c in CHECKS if c["verdict"] == "WARN"]
    print(f"\nValidation summary: {len(CHECKS)} checks — {len(CHECKS) - len(failed) - len(warned)} PASS, {len(warned)} WARN, {len(failed)} FAIL")
    if failed:
        raise RuntimeError("Validation failed: " + "; ".join(f"{c['check']} ({c['detail']})" for c in failed))

## 13. Framing your differentiator — BigQuery ML (`CREATE MODEL` + `ML.EXPLAIN_PREDICT`)

This is where the starter notebook stops and your team's modeling and agent design take over.

### Step A — Build a leakage-free, race-free training view (`v_tract_training_features`)

Notice what is **deliberately excluded** from this view:
1. `routine_checkup_within_1yr_pct` and `state_mean_checkup_pct` (target leakage — Section 2),
2. `satellite_only_bsl_count` (exact collinear duplicate of `unserved_bsl_under_25_3_count`), and
3. All five `audit_*` demographic columns (Section 11 Equity Audit rule).

```sql
CREATE OR REPLACE VIEW `a4i_telehealth.v_tract_training_features` AS
WITH clinic_dist AS (
  SELECT
    a.census_tract_geoid,
    ROUND(MIN(ST_DISTANCE(ST_CENTROID(a.tract_geom), f.facility_geom)) / 1000.0, 2)
      AS nearest_active_clinic_km,
    ROUND(MIN(IF(f.is_statutory_safety_net_clinic,
                 ST_DISTANCE(ST_CENTROID(a.tract_geom), f.facility_geom), NULL)) / 1000.0, 2)
      AS nearest_safety_net_km
  FROM `a4i_telehealth.acs_income_and_device_ownership` AS a
  INNER JOIN `a4i_telehealth.active_health_facilities_registry` AS f
    ON a.state_code = f.state_code
  GROUP BY 1
)
SELECT
  u.census_tract_geoid,
  u.state_code,
  b.rural_urban_context,
  u.checkup_gap_pct_pts,
  ROUND(SAFE_DIVIDE(b.unserved_bsl_under_25_3_count, b.total_bsl_locations), 4) AS unserved_bsl_share,
  ROUND(SAFE_DIVIDE(a.hh_no_internet_access_count, a.total_households), 4) AS no_internet_share,
  ROUND(SAFE_DIVIDE(a.hh_cellular_data_only_count, a.total_households), 4) AS cellular_only_share,
  ROUND(SAFE_DIVIDE(a.hh_no_computing_device_count, a.total_households), 4) AS no_device_share,
  ROUND(SAFE_DIVIDE(a.hh_no_vehicle_count, a.total_households), 4) AS no_vehicle_share,
  a.poverty_rate,
  u.uninsured_adults_pct,
  d.nearest_safety_net_km
FROM `a4i_telehealth.cdc_tract_health_utilization` AS u
INNER JOIN `a4i_telehealth.acs_income_and_device_ownership` AS a USING (census_tract_geoid)
INNER JOIN `a4i_telehealth.fcc_broadband_coverage_by_tract` AS b USING (census_tract_geoid)
INNER JOIN clinic_dist AS d USING (census_tract_geoid);
```

### Step B — Train an interpretable BQML model & generate per-tract driver attributions

`LINEAR_REG` trains in ~15 seconds on BigQuery's free/on-demand tier and supports exact local feature attributions via `ML.EXPLAIN_PREDICT`:

```sql
CREATE OR REPLACE MODEL `a4i_telehealth.checkup_gap_model`
OPTIONS (
  model_type = 'LINEAR_REG',
  input_label_cols = ['checkup_gap_pct_pts'],
  enable_global_explain = TRUE
) AS
SELECT * EXCEPT (census_tract_geoid, state_code)
FROM `a4i_telehealth.v_tract_training_features`;

-- Inspect model fit first (if R^2 = 1.000, you leaked a checkup column!)
SELECT * FROM ML.EVALUATE(MODEL `a4i_telehealth.checkup_gap_model`);

-- Generate per-tract top-3 feature attributions for your planner agent
SELECT
  census_tract_geoid,
  rural_urban_context,
  ROUND(checkup_gap_pct_pts, 2) AS actual_gap_pts,
  ROUND(predicted_checkup_gap_pct_pts, 2) AS predicted_gap_pts,
  top_feature_attributions
FROM ML.EXPLAIN_PREDICT(
  MODEL `a4i_telehealth.checkup_gap_model`,
  (SELECT * FROM `a4i_telehealth.v_tract_training_features`),
  STRUCT(3 AS top_k_features)
)
ORDER BY predicted_checkup_gap_pct_pts DESC
LIMIT 10;
```

### Five traps that will cost you credit if you miss them

1. **Predicting the gap is not prescribing the intervention.** Two tracts with the same `+3.5` pt checkup gap need opposite budgets if Tract A's top driver is `unserved_bsl_share` (`<25/3 Mbps` infrastructure gap -> fiber/kiosk + mobile van) while Tract B's top driver is `no_device_share` in a fiber-served neighborhood (-> cellular-enabled tablet lending kit).
2. **`R^2 = 1.00` is a bug, not a victory.** Always show `ML.EVALUATE` and `ML.WEIGHTS` / `top_feature_attributions` so judges can see real, non-leaked drivers.
3. **Sample size vs. model complexity:** A single state has `43–101` tracts. Deep boosted trees (`BOOSTED_TREE_REGRESSOR`) take several minutes to train and readily overfit 50 rows unless heavily regularized (`max_tree_depth = 2`, `subsample = 0.8`) or trained across a multi-state feature table (`428` rows). Start with `LINEAR_REG` first.
4. **Telehealth cannot replace urgent/inpatient care after a hospital closure.** When `historical_facility_closures_10yr` shows a recent acute hospital closure nearby and households also lack vehicles (`no_vehicle_rate`), flag the need for mobile primary care or non-emergency transport alongside digital investments.
5. **Audit *after* predicting.** Join `acs_income_and_device_ownership`'s `audit_*` columns to your final intervention allocation table and report whether any underserved community was systematically assigned lower-tier support.

### Four questions judges will ask from your 3-minute video & deck

1. **Intervention matching:** Show two high-gap tracts where `ML.EXPLAIN_PREDICT` surfaces *different* top attributions and your agent recommends *different* concrete interventions.
2. **Leakage & collinearity hygiene:** How did you verify your BQML model was free of target leakage, collinear duplicates, and `audit_*` race features?
3. **Real vs. modeled honesty:** Does your presentation state clearly that CDC PLACES checkup rates, ACS device counts, and organizational CMS NPPES facilities are empirical public records while `fcc_broadband_coverage_by_tract` is `[Modeled]` from FCC BDC benchmarks?
4. **Double-jeopardy insight:** What happens in your state's rural tracts near a 10-year facility closure when you try to solve healthcare loss with video visits alone?

## 14. Diagnostic summary

In [ ]:
print("=" * 72)
print("A4I DIGITAL & TELEHEALTH EQUITY — DIAGNOSTIC SUMMARY")
print("=" * 72)
print(f"project     : {PROJECT}")
print(f"dataset     : {PROJECT}.{DATASET} ({LOCATION})")
print(f"state       : {STATE_NAME} ({STATE}, state FIPS {STATE_FIPS})")
print("-" * 72)
print("TABLES")
for tbl_item in bq.list_tables(f"{PROJECT}.{DATASET}"):
    tbl_obj = bq.get_table(tbl_item)
    cols_preview = ", ".join(f"{f.name}:{f.field_type}" for f in tbl_obj.schema[:6])
    print(f"  {tbl_obj.table_id:<38} {tbl_obj.num_rows:>6,} rows  [{cols_preview}, ...]")
print("-" * 72)
print("CHECKS")
for ck in CHECKS:
    print(f"  {ck['verdict']:<5} {ck['check']}: {ck['detail']}")
print("-" * 72)
print("TIMING")
for s_name, secs in STEPS.items():
    print(f"  {s_name:<44} {secs:>6.1f}s")
print(f"  {'TOTAL WALL CLOCK':<44} {time.time() - NOTEBOOK_START:>6.1f}s")
print("=" * 72)